# Lab 5 — Structured Streaming (local version)

1. Landing zone and generator
2. First file stream
3. Checkpoint recovery
4. Schema evolution
5. Event time vs processing time
6. Watermarked windows
7. Monitoring

**Stop every stream before you leave.** The last cell does it.

This lab is for self-study: every question is followed by a collapsed **Answer**. Commit to your own answer (and, for the experiments, a prediction) before you open it.

---

This is the local port of `L5_streaming.py`, which was written for Databricks. Every section runs
locally. Wherever Databricks behaves differently, and not just with different syntax, a
**☁️ On Databricks** box after the section says what you would see there. Smaller swaps are kept
as `# Databricks:` comments next to the local code:

| Databricks | Here |
|---|---|
| Unity Catalog schema `workspace.lab_<user>` | local database `lab5` in `spark-warehouse/` |
| Volume `/Volumes/.../stream` | folder `data/l5/` (git-ignored) |
| `dbutils.fs.*` | `pathlib` / `shutil` |
| Auto Loader (`format("cloudFiles")`) | built-in file source (`format("json")`) with an explicit schema |
| `cloudFiles.schemaLocation`, `schemaHints`, schema evolution | not available — we change the schema by hand (section 4) |
| `_rescued_data` column for values that do not fit the schema | none: such values are silently dropped or set to `NULL` (section 2) |
| `display(df)` | `df.show()` |
| streaming dashboard under the cell | *Structured Streaming* tab of the Spark UI (<http://localhost:4040>) + `monitor_dashboard.py` (Streamlit, section 7) |

## Setup

In [ ]:
import shutil

from pyspark_demo.generate_batch import (
    CKPT,  # path to checkpoints
    LANDING,  # path to landing zone
    ROOT_PATH,  # repo root path
    STREAMING_BASE,  # path to root folder for streaming data
)

WAREHOUSE = ROOT_PATH / "spark-warehouse"
DB = "lab5"

# prepare streaming data folder
shutil.rmtree(STREAMING_BASE, ignore_errors=True)
shutil.rmtree(WAREHOUSE / f"{DB}.db", ignore_errors=True)
LANDING.mkdir(parents=True)
print(f"landing = {LANDING}")
print(f"warehouse = {WAREHOUSE}")


In [ ]:
import time

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, LongType, StringType, StructField, StructType

builder = (
    SparkSession.builder.appName("L5 streaming")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.warehouse.dir", str(WAREHOUSE))
    # The generator writes UTC timestamps; make Spark read and print them as UTC as well.
    .config("spark.sql.session.timeZone", "UTC")
    # 200 shuffle partitions = 200 state stores per stateful batch; far too many on a laptop.
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", False)  # it is not supported in streaming DataFrames/Datasets
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark.sql(f"DROP DATABASE IF EXISTS {DB} CASCADE")
spark.sql(f"CREATE DATABASE {DB}")
spark.sql(f"USE {DB}")

## 1. The generator

There is no real sensor fleet, so `drop_batch()` plays one. It lives in
[`pyspark_demo/generate_batch.py`](../../pyspark_demo/generate_batch.py). Each call:

1. Builds `n_records` readings for 10 devices (`device_00` … `device_09`): `device_id`,
   `event_time`, `temperature` (normally distributed around 20 °C), and `humidity`.
2. Stamps `event_time` with **the current UTC time minus an offset**. Normally the offset is
   under 30 seconds. For a `late_fraction` share of readings it is up to `late_minutes` (45 by
   default), which is what a device does when it buffers offline and uploads later. The timestamp is
   ISO-8601 text with no time zone, which is why Setup sets the session time zone to UTC.
3. With `extra_field=True`, adds a `battery_pct` field. Section 4 uses this as the "firmware update".
4. Writes the batch as **one JSON-lines file** (one reading per line) into `data/l5/landing/`, and
   returns its path.

The write is done in two steps: first to a hidden `.batch_….json.tmp` file, then a rename to
`batch_<nanoseconds>.json`. A streaming file source can list the folder at any moment. If it caught a
file halfway through being written, it would read a truncated batch and never look at the file again.
A rename is atomic, and Spark ignores names starting with `.` or `_`, so the stream only ever sees
complete files. Real producers need the same discipline.

In [ ]:
from pyspark_demo.generate_batch import drop_batch  # helper method to generate data

path = drop_batch(200)
print(path)
with open(path) as f:
    print(f.readline())  # one line = one reading
print(f"{len(list(LANDING.glob('*.json')))} file(s) in landing")

> **On Databricks** the landing zone is a Unity Catalog **Volume**
> (`/Volumes/<catalog>/<schema>/stream/landing`), and the generator writes each file with a single
> `dbutils.fs.put(path, content)`. A Volume write only becomes visible once the file is complete, so the
> temp-file-and-rename trick is not needed. In production nobody calls `drop_batch()`: files arrive in
> cloud storage from the producers, such as an IoT hub export, Kafka Connect or a partner's SFTP drop.
> For large directories, Auto Loader can then be switched from listing the folder to **file
> notification** mode (`cloudFiles.useNotifications`), where the cloud's storage events tell it
> about new files.

## 2. Your first file stream

**Locally** we use Spark's built-in **file source**: `spark.readStream.format("json")`. It
remembers the files it has processed in the checkpoint, so a restart does not reprocess them.
However:
- it needs the schema **up front** (there's no inference for streams, no `schemaHints`, and no
  evolution);
- it **re-lists the whole directory** on every micro-batch, so it slows down as the folder grows;
- anything in a file that does not fit the schema is silently lost (see below).

> **On Databricks** this section uses **Auto Loader**: `spark.readStream.format("cloudFiles")`.
> It infers the schema from the first files and saves it in `cloudFiles.schemaLocation`, lets you
> pin the types of some columns with `cloudFiles.schemaHints`, and evolves the schema when new
> fields appear (section 4). It keeps the files it has seen in a RocksDB store inside the checkpoint,
> so a growing directory costs nothing extra. The Databricks version of the read is in the comment in
> the next cell.

In [ ]:
readings_schema = StructType(
    [
        StructField("device_id", StringType()),
        StructField("event_time", StringType()),  # a string here is on purpose
        StructField("temperature", DoubleType()),
        StructField("humidity", DoubleType()),
    ]
)


def bronze_source(schema: StructType):
    # Databricks:
    #   spark.readStream
    #   .format("cloudFiles")
    #   .option("cloudFiles.format", "json")
    #   .option("cloudFiles.schemaLocation", f"{SCHEMA_LOC}/bronze")
    #   .option("cloudFiles.schemaHints", "temperature double, humidity double")
    #   .load(LANDING)

    return (
        spark.readStream.format("json")
        .schema(schema)
        .load(str(LANDING))
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_ingested_at", F.current_timestamp())
    )


bronze_stream = bronze_source(readings_schema)
#

print("isStreaming:", bronze_stream.isStreaming)
bronze_stream.printSchema()

### Local vs Databricks: what `printSchema()` shows

Locally, the schema is the `StructType` we passed in, plus the two columns we added (the `nullable` flags are left out here):

```
root
 |-- device_id: string
 |-- event_time: string
 |-- temperature: double
 |-- humidity: double
 |-- _source_file: string
 |-- _ingested_at: timestamp
```

> **On Databricks** the same cell prints a schema that Auto Loader **inferred**, plus one column
> you never asked for:
>
> ```
> root
>  |-- device_id: string
>  |-- event_time: string
>  |-- humidity: double           <- from schemaHints; everything else inferred as string
>  |-- temperature: double        <- from schemaHints
>  |-- _rescued_data: string      <- added by Auto Loader
>  |-- _source_file: string
>  |-- _ingested_at: timestamp
> ```
>
> **`_rescued_data`** is Auto Loader's safety net. It holds, as a JSON string, any value that did not
> fit the schema: a field with the wrong type (`"temperature": "n/a"`), a field whose name differs only
> in case, and, in `rescue` mode, fields the schema doesn't know about. Nothing is lost; it is
> parked in that column where you can count it and fix it. Note also that on Databricks, `printSchema()`
> is not free: inferring the schema means Auto Loader has already **read a sample of the files**.

**Locally there is no equivalent.** The plain file source handles a mismatch in one of two ways:
- a field that is **not in the schema** is dropped without a trace (section 4 shows this);
- a value of the **wrong type** becomes `NULL` in `PERMISSIVE` mode, which is the default.

The nearest local tool is `.option("columnNameOfCorruptRecord", "_corrupt_record")` plus a
`_corrupt_record` string field in the schema. That keeps the whole raw line when a line cannot be
parsed, but it still ignores unknown fields.

### What the two extra columns are

- `_metadata` is a **hidden column** that every file-based source (JSON, CSV, Parquet, …) exposes. It
  never shows up in `printSchema()` or `SELECT *` until you ask for it. It is a struct with
  `file_path`, `file_name`, `file_size`, `file_block_start`, `file_block_length` and
  `file_modification_time`. Copying `file_path` into `_source_file` gives every bronze row its
  **lineage**: when a reading looks wrong, you can go straight to the file it came from. It also lets
  you prove, in Q3b, which files were ingested twice.
- `_ingested_at = current_timestamp()` is **processing time**. In a streaming query it is evaluated
  once per micro-batch, not once per row, so every row of a batch gets the same value (check the
  output below). That makes it a batch marker, which section 5 relies on.

By convention, columns that the pipeline adds, as opposed to fields that came from the source, start
with `_`.

### What has actually happened so far

Locally, nothing has been read. `spark.readStream...load()` only built a **plan** for an unbounded
DataFrame: "whatever JSON files are, or will be, in `LANDING`, with this schema". That is why
`printSchema()` answers instantly (it echoes our `StructType` plus the two new columns), and why
`isStreaming` is `True`. No file has been listed, no job has run, and nothing appears in the Spark UI
yet. You cannot call `.count()` or `.show()` on it; try it, and Spark tells you to use
`writeStream.start()`.

The next cell starts a query. With `trigger(availableNow=True)`, it does the following:

1. It lists `LANDING` and compares the listing with the checkpoint's file log (`sources/0/`), which
   is empty on the first run, so every file is new.
2. It writes the plan for batch 0 to `offsets/0`. This is the write-ahead log.
3. It reads those files, adds the two columns, and appends the rows to the Delta table
   `bronze_readings`, creating the table on the first run.
4. It records the batch as done in `commits/0`.
5. It finds nothing left to read and stops. `awaitTermination()` returns.

Section 3 looks inside that checkpoint.

In [ ]:
q_bronze = (
    bronze_stream.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", str(CKPT / "bronze"))
    .trigger(availableNow=True)  # drain what is there, then stop
    .toTable("bronze_readings")
)
q_bronze.awaitTermination()

print(f"bronze_readings: {spark.table('bronze_readings').count():,} rows")

# Databricks: display(spark.table("bronze_readings").limit(5))
spark.table("bronze_readings").show(5, truncate=False)

**Q2.** We declared `event_time` as a `StringType`, even though every value is an ISO-8601 timestamp.
What would change if we declared it `TimestampType()` instead? In each case, what happens to a reading
whose `event_time` is `"not-a-time"`, and to one whose `temperature` is `"n/a"`? In which layer would you
convert `event_time`, and why?

<details>
<summary><b>Answer</b></summary>

**Declared as `TimestampType()`**, Spark parses the value while it reads the file. Bronze gets a real
timestamp column, and sections 5 and 6 would not need `F.to_timestamp`. The price is paid on bad
values:

| value in the file | `event_time` as `StringType` (as here) | `event_time` as `TimestampType` |
|---|---|---|
| `"2026-09-29T10:00:00"` | `"2026-09-29T10:00:00"` | `2026-09-29 10:00:00` |
| `"not-a-time"` | `"not-a-time"`, kept as is | `NULL`: **the original text is gone** |

Neither choice raises an error. In the default `PERMISSIVE` mode, Spark keeps the reading and sets
the field it cannot parse to `NULL`. `temperature: "n/a"` shows this: we declared it `DoubleType`,
so it becomes `NULL` **silently**, with nothing left to show that the file said `"n/a"`.

**Where to convert: in silver.** Keep bronze as a faithful copy of the source (string), and parse
in the next layer, as sections 5 and 6 do with `F.to_timestamp`. Two reasons:
- a bad value can never break or distort ingestion, and the raw text is still there to debug or
  re-parse (for example, if the producer switches to a different date format);
- you can **measure** bad values: `event_time IS NOT NULL AND event_ts IS NULL` counts the rows
  that failed to parse. A typed bronze column turns these into `NULL`s you can't tell apart from
  genuinely missing values.

</details>

> **☁️ On Databricks** the same choice comes up differently. Auto Loader types every JSON column as a
> string by default, unless you name it in `cloudFiles.schemaHints` or set `cloudFiles.inferColumnTypes`.
> So `event_time` arrives as a string without you choosing it. Values that don't fit a declared type,
> like `"n/a"` for a `double`, are not lost there: Auto Loader parks them in `_rescued_data`
> (section 2).

## 3. Checkpoint recovery

Drop three more batches, then run the same stream again. It must pick up **only**
the new files.

In [ ]:
before = spark.table("bronze_readings").count()
for _ in range(3):
    drop_batch(200)

q = (
    bronze_stream.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", str(CKPT / "bronze"))
    .trigger(availableNow=True)
    .toTable("bronze_readings")
)
q.awaitTermination()

after = spark.table("bronze_readings").count()
print(f"{before:,} -> {after:,}  (+{after - before:,})")

In [ ]:
# CHECK
assert after - before == 600, f"expected exactly 600 new rows, got {after - before}"
print("The file source processed only the new files. No duplicates.")

**Q3a.** Look inside the checkpoint directory. What is in `offsets/`, and what is in
`commits/`? Why are they separate?

(You will also see `sources/0/`. **Locally** it is the file source's log of every file it has
processed: plain text files, one per batch, compacted every 10 batches.
**☁️ On Databricks** the same path holds Auto Loader's **RocksDB** store instead.
Looking up "have I seen this file?" there takes constant time, however many files the folder
holds.)

<details>
<summary><b>Answer</b></summary>

- **`offsets/<N>`**: written **before** batch *N* runs. It is the write-ahead log that says
  exactly what batch *N* will read. For the file source, that is the id of the entry in `sources/0/`
  listing the batch's files. It also stores the batch's event-time watermark, its timestamp, and the
  session confs that must not change between runs, such as the number of shuffle partitions.
- **`commits/<N>`**: written **after** the sink has durably written batch *N*. It is the "done"
  marker, and it also carries the watermark for the next batch.
- **`sources/0/`**: the file source's log of processed files, one entry per batch. Every 10
  batches it is compacted into a `.compact` file.
- **`metadata`**: the query's permanent id.

**Why the two logs are separate.** They bracket the batch, so after a crash Spark can tell which
case it is in:

| state after a crash | what Spark does on restart |
|---|---|
| `offsets/N` without `commits/N` | re-runs batch *N* on **exactly the same files** it planned |
| both present | batch *N* is done, so it plans *N+1* from new files |

Re-running batch *N* would duplicate rows if the sink wrote them again. The Delta sink prevents that:
it records the query id and batch id in the table's transaction log, and it skips a batch it has
already committed. Replayable input (`offsets`) plus an idempotent sink is how Structured Streaming
achieves **exactly-once** delivery. With a single log you would not know whether a crashed batch
reached the sink.

</details>

In [ ]:
# Databricks: for d in dbutils.fs.ls(f"{CKPT}/bronze"): print(d.name)
for d in sorted((CKPT / "bronze").iterdir()):
    print(d.name + ("/" if d.is_dir() else ""))

**Reading the logs.** If you open `offsets/` in a file browser, the first thing you see is
`.0.crc`, `.1.crc`, … These are binary **checksum sidecars**: Hadoop's local file system writes one
next to every file so it can detect corruption. They are not the log. The log entries are the files
named `0`, `1`, `2`, …, and they are plain text: a format version line (`v1`) followed by JSON.

Follow the chain for one batch *N*:
1. `offsets/N` contains `{"logOffset": N}`.
2. `sources/0/N` lists the exact files batch *N* read.
3. `commits/N` exists only if batch *N* finished.

> **☁️ On Databricks** the checkpoint lives in cloud storage, which has no `.crc` sidecars, so a
> directory listing shows only the log files. They have the same `v1` + JSON format, and you can read
> one with `dbutils.fs.head(f"{CKPT}/bronze/offsets/0")`. The source part of an `offsets` entry is
> Auto Loader's own offset, not a `logOffset`, and `sources/0/` holds its RocksDB store rather than
> readable file lists.

In [ ]:
from pyspark_demo.show_log import show_log

for log in ["offsets", "commits", "sources/0"]:
    show_log(CKPT / "bronze" / log)
print("== metadata:", (CKPT / "bronze" / "metadata").read_text().strip())

**Q3b.** Delete the checkpoint directory and re-run the stream. Predict what happens
first, then do it, then explain what you saw. (Drop the table afterwards so the rest
of the notebook is clean.)

<details>
<summary><b>Answer</b></summary>

**Prediction and result.** The checkpoint is the stream's **only** memory. Without it, the restarted
query is a brand-new query: an empty `sources/0/` means every file in `LANDING` counts as new. All
4 files are read again and appended, so the row count doubles (800 becomes 1,600), and each file shows
`times_ingested = 2`.

**Why Delta did not catch it.** The Delta sink's duplicate protection from Q3a is keyed on the
query id **and** the batch id. Deleting the checkpoint also deleted the query id (`metadata`), so the
new run arrives as a different query writing "batch 0" again. As far as Delta knows, that is
legitimate new data.

**Lessons:**
- The checkpoint is **part of the table's state**, not a cache. Back it up with the table and never
  delete it to "unstick" a stream. You will get silent duplicates, and a bronze table has no key to
  deduplicate on.
- To reprocess on purpose, reset **both** the checkpoint and the target, as the reset cell does, or
  write to a new table.
- If the target must survive a checkpoint loss, make it idempotent: `foreachBatch` + `MERGE` on a
  natural key, or deduplicate downstream on (`_source_file`, row position).

</details>

In [ ]:
# Q3b. Prediction first, then run.


def ingest_bronze():
    q = (
        bronze_stream.writeStream.format("delta")
        .outputMode("append")
        .option("checkpointLocation", str(CKPT / "bronze"))
        .trigger(availableNow=True)
        .toTable("bronze_readings")
    )
    q.awaitTermination()


n_before = spark.table("bronze_readings").count()

shutil.rmtree(CKPT / "bronze")  # the experiment: the stream forgets everything it has read
ingest_bronze()

n_after = spark.table("bronze_readings").count()
print(f"rows: {n_before:,} -> {n_after:,}")

# Every file now appears twice, once per ingestion run (a run shares one _ingested_at).
(
    spark.table("bronze_readings")
    .groupBy(F.element_at(F.split("_source_file", "/"), -1).alias("file"))
    .agg(F.count("*").alias("rows"), F.countDistinct("_ingested_at").alias("times_ingested"))
    .orderBy("file")
    .show(truncate=False)
)

In [ ]:
# Reset: drop the table, delete the checkpoint again, and ingest from scratch, once.
spark.sql("DROP TABLE IF EXISTS bronze_readings")
shutil.rmtree(WAREHOUSE / f"{DB}.db" / "bronze_readings", ignore_errors=True)
shutil.rmtree(CKPT / "bronze", ignore_errors=True)
ingest_bronze()

n_reset = spark.table("bronze_readings").count()
assert n_reset == n_before, f"expected {n_before:,} rows after the reset, got {n_reset:,}"
print(f"clean again: {n_reset:,} rows, each file ingested once")

> **☁️ On Databricks** the experiment behaves the same, because the checkpoint is the stream's
> only memory there too. The code differs only in the tools: `dbutils.fs.rm(f"{CKPT}/bronze",
> recurse=True)` instead of `shutil.rmtree`. Dropping a Unity Catalog managed table removes its
> files for you, so there is no warehouse folder to clean up.

## 4. Schema evolution

The device firmware is updated and starts reporting `battery_pct`. This is the realistic case where
the source changes its data contract and forgets to tell the people who consume it.

**Locally** the file source has no schema evolution: its schema is whatever you passed in. Watch what
that means.

> **☁️ On Databricks** Auto Loader, in its default `schemaEvolutionMode = addNewColumns`, notices
> the new field. It records the widened schema in `schemaLocation` and **stops** the stream with an
> `UnknownFieldException`. The failed batch is never committed, so when you re-run the cell (or a job
> retries), the restarted stream re-reads the **same file** with the new schema. `battery_pct` is
> kept for every reading, and nothing is lost. The other modes are `rescue` (the schema never
> changes; new fields go into `_rescued_data`), `failOnNewColumns` (stop until a human changes the
> schema), and `none` (ignore new fields). `none` is what the local run below behaves like.

In [ ]:
drop_batch(200, extra_field=True)

# Same code as on Databricks, but here it does not stop: the new field is not in
# readings_schema, so it is silently dropped (Auto Loader's schemaEvolutionMode = "none").
try:
    q = (
        bronze_stream.writeStream.format("delta")
        .outputMode("append")
        .option("checkpointLocation", str(CKPT / "bronze"))
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable("bronze_readings")
    )
    q.awaitTermination()
    print("stream completed")
except Exception as e:
    print(type(e).__name__)
    print(str(e)[:500])

print("columns:", spark.table("bronze_readings").columns)

No error, no `battery_pct`. To get the column we do by hand what Auto Loader does for us:
widen the schema and restart on the **same** checkpoint. `mergeSchema` lets the Delta sink
add the column to the table.

Files already processed are not re-read, so the 200 readings from the batch above keep
`battery_pct = NULL` forever. **☁️ On Databricks** they would have kept their `battery_pct`: the
stream stops *before* committing the batch, rather than after silently dropping the field.

In [ ]:
readings_schema_v2 = StructType(readings_schema.fields + [StructField("battery_pct", LongType())])
bronze_stream = bronze_source(readings_schema_v2)  # every later section uses the new schema

drop_batch(200, extra_field=True)

q = (
    bronze_stream.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", str(CKPT / "bronze"))
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("bronze_readings")
)
q.awaitTermination()

(spark.table("bronze_readings").groupBy(F.col("battery_pct").isNotNull().alias("has_battery_pct")).count().show())

**Q4.** In section 4 the stream lost `battery_pct` for 200 readings, and nothing failed or even
warned. (a) Why did Spark not complain? (b) How would anyone have found out? (c) Sketch how to make this
local pipeline either **fail loudly** on an unknown field, or **keep** it, instead of dropping it
silently.

<details>
<summary><b>Answer</b></summary>

**(a) Why nothing failed.** The file source applies a schema *on read*: the JSON parser fills the
fields it was asked for and ignores every other key. Unknown fields are not an error in any parse
mode. `PERMISSIVE`, `DROPMALFORMED` and `FAILFAST` only react to lines that *cannot be parsed*,
and a line with an extra key parses perfectly. The batch succeeded, the checkpoint recorded the file
as processed, and it will never be read again.

**(b) How you would find out.** Only by comparing with the source. For example, a scheduled batch
read of the landing folder with schema inference (`spark.read.json(str(LANDING)).columns`) compared
with the bronze columns, or a producer who announces the change. In practice it is found weeks later
by an analyst asking why `battery_pct` is missing. The data needed to backfill is still in the
files, but the stream will not look at them again.

**(c) Fail loudly, or keep it.** Read the files as **text**, parse with `from_json`, and compare
each line's keys with the schema:

```python
expected = F.array(*[F.lit(f.name) for f in readings_schema.fields])

checked = (
    spark.readStream.format("text").load(str(LANDING))
    .withColumn("r", F.from_json("value", readings_schema))
    .withColumn("unknown_fields", F.array_except(F.json_object_keys("value"), expected))
    .select(
        "r.*",
        # policy "rescue": keep the whole raw line whenever it carries something unexpected
        F.when(F.size("unknown_fields") > 0, F.col("value")).alias("_rescued_data"),
        "unknown_fields",
    )
)


def fail_on_new_fields(batch_df, batch_id):  # policy "fail": stop before anything is written
    new = batch_df.select(F.explode("unknown_fields")).distinct().collect()
    if new:
        raise ValueError(f"batch {batch_id}: unknown fields {[r[0] for r in new]}. Update the schema and restart.")
    batch_df.drop("unknown_fields").write.format("delta").mode("append").saveAsTable("bronze_readings")


checked.writeStream.foreachBatch(fail_on_new_fields).option("checkpointLocation", ...).start()
```

- **Fail**: the exception stops the stream **before** the batch is committed, so the file is not
  marked as processed. After you widen the schema and restart, the same file is read again, and
  `battery_pct` is kept for every reading.
- **Rescue**: drop the `raise`, and the unknown fields survive in `_rescued_data`. You alert on
  `_rescued_data IS NOT NULL` and promote the field to a real column when you are ready.

(We tested both policies on a file containing a `battery_pct` line.)

</details>

> **☁️ On Databricks** you get this protection without writing it. Auto Loader's `schemaEvolutionMode`
> offers these policies as settings: `addNewColumns` (the default: stop, record the new column, and
> continue after a restart), `failOnNewColumns` (stop until a human changes the schema), `rescue`
> (keep unknown fields in `_rescued_data`), and `none` (drop them, like the local run above). The
> Databricks version of this lab asks you to argue for the default instead.

## 5. Event time vs processing time

To simulate 'late' records we use drop_batch where 40% of records are up to 45 minutes late, and aggregate the
same data two ways to show the effect. 

In [ ]:
drop_batch(500, late_fraction=0.4, late_minutes=45)

q = (
    bronze_stream.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", str(CKPT / "bronze"))
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("bronze_readings")
)
q.awaitTermination()

readings = spark.table("bronze_readings").withColumn("event_ts", F.to_timestamp("event_time"))
print(f"{readings.count():,} rows")

In [ ]:
by_processing = (
    readings.groupBy(F.window("_ingested_at", "10 minutes").alias("w"))
    .agg(F.count("*").alias("n"), F.avg("temperature").alias("avg_temp"))
    .select(F.col("w.start").alias("window_start"), "n", "avg_temp")
    .orderBy("window_start")
)

by_event = (
    readings.groupBy(F.window("event_ts", "10 minutes").alias("w"))
    .agg(F.count("*").alias("n"), F.avg("temperature").alias("avg_temp"))
    .select(F.col("w.start").alias("window_start"), "n", "avg_temp")
    .orderBy("window_start")
)

print("--- by processing time ---")
# Databricks: display(by_processing)
by_processing.show(truncate=False)

print("--- by event time ---")
# Databricks: display(by_event)
by_event.show(truncate=False)

**Q5.** How many windows does each grouping produce? Which is the truthful answer to
"what was the average temperature between 14:00 and 14:10", and why is the other one
wrong in a way that would not be obvious to whoever reads the dashboard?

<details>
<summary><b>Answer</b></summary>

**Window counts.** Processing time gives **one window per 10 minutes of wall-clock time in which
you ran the ingestion cells**. `_ingested_at` is fixed per micro-batch, so all the rows of one run
share a single timestamp. Run the notebook straight through and everything lands in **1** window;
spread the sections over half an hour and you get 3 or 4. Event time gives **one window per 10
minutes of real measurements**: 5–6 windows spanning the 45 minutes that the late readings cover,
however fast you ran the cells.

**The truthful one is event time.** The question asks about temperatures *measured* between 14:00
and 14:10, and only `event_ts` records when a measurement was taken. The processing-time grouping
answers a different question: "what did the pipeline receive between 14:00 and 14:10?" It files a
reading taken at 14:20 but delivered at 15:05 under 15:00, mixing measurements taken up to 45 minutes
apart, and it may show nothing at all for 14:00–14:10.

**Why nobody would notice.** The wrong numbers look perfectly plausible. Averages are around 20 °C
either way, counts are positive, and there are no nulls or errors. The chart just has fewer, fatter
bars. Worse, the numbers depend on **when the pipeline ran**: a delay, a backfill or a re-run moves
rows between windows, so yesterday's dashboard changes without the data changing. Event-time
results depend only on the data, so they are reproducible.

</details>

## 6. Watermarked windowed aggregation

'Late' records does have an effect during the aggregation. And this is another common scenario.
For example, customer made a purchase two days ago but we only received transaction data today. 
The effect of data engineer decision at this point will be at gold layer when aggregation are made. 

A streaming aggregation with no watermark keeps every window's state forever. Add one.

In [ ]:
silver_stream = (
    spark.readStream.table("bronze_readings")
    .withColumn("event_ts", F.to_timestamp("event_time"))
    # Watermark = max(event_ts) seen so far - 15 minutes. Readings are accepted up to 15 minutes late;
    # a window whose end falls behind the watermark is final: it is emitted and its state dropped,
    # and any reading for it that arrives later is discarded.
    .withWatermark("event_ts", "15 minutes")
    .groupBy(F.window("event_ts", "10 minutes"), "device_id")
    .agg(
        F.count("*").alias("readings"),
        F.avg("temperature").alias("avg_temp"),
        F.max("temperature").alias("max_temp"),
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "device_id",
        "readings",
        "avg_temp",
        "max_temp",
    )
)

q_silver = (
    silver_stream.writeStream.format("delta")
    .outputMode("append")  # legal only BECAUSE of the watermark
    .option("checkpointLocation", str(CKPT / "silver"))
    .trigger(availableNow=True)
    .toTable("silver_readings_10min")
)
q_silver.awaitTermination()

# Databricks: display(spark.table("silver_readings_10min").orderBy("window_start", "device_id"))
spark.table("silver_readings_10min").orderBy("window_start", "device_id").show(50, truncate=False)

> **☁️ On Databricks** the aggregation code is identical. The difference is where the per-window
> state lives. Locally it is the default in-memory state store (**HDFS-backed**), snapshotted to
> `checkpoints/silver/state/`. That is fine for 10 devices, but all the state has to fit in memory.
> Databricks recommends the **RocksDB** state store for any real volume. It keeps state on local disk
> and scales to millions of keys:
> `spark.conf.set("spark.sql.streaming.stateStore.providerClass",
> "com.databricks.sql.streaming.state.RocksDBStateStoreProvider")`. Open-source Spark ships a
> RocksDB provider too, under `org.apache.spark.sql.execution.streaming.state`, so you can try it
> locally.

**Q6a.** Remove `.withWatermark(...)` from the cell above and re-run it into a
*different* table and checkpoint. What error do you get, and what exactly is Spark
objecting to?

<details>
<summary><b>Answer</b></summary>

The query fails **before it starts**, at analysis time: nothing is read and no table is created.

```
AnalysisException: [STREAMING_OUTPUT_MODE.UNSUPPORTED_OPERATION] Invalid streaming output mode: append.
This output mode is not supported for streaming aggregations without watermark on streaming DataFrames/DataSets.
```

**What Spark objects to.** `append` promises that every output row is written **once and never
changed**. An aggregate over a window can change whenever another reading for that window arrives,
and without a watermark a reading can arrive for *any* window at *any* time. So Spark can never
declare a window final, which means it could never emit a single row, and it would have to keep
every window's state forever. The contract can't be met, so Spark rejects the query up front
instead of running one that outputs nothing and slowly runs out of memory.

The ways out:
- **add a watermark** (section 6), which defines when a window is final;
- use **`complete`** mode, which rewrites the entire result on every batch (supported by the Delta
  sink, but the state and the output grow forever);
- use **`update`** mode, which emits windows that changed in this batch (not supported by the Delta
  sink directly; you need `foreachBatch` + `MERGE`).

</details>

In [ ]:
# Q6a. The same aggregation without .withWatermark(...), into its own table and checkpoint.
no_watermark = (
    spark.readStream.table("bronze_readings")
    .withColumn("event_ts", F.to_timestamp("event_time"))
    .groupBy(F.window("event_ts", "10 minutes"), "device_id")
    .agg(F.count("*").alias("readings"), F.avg("temperature").alias("avg_temp"))
)

try:
    q = (
        no_watermark.writeStream.format("delta")
        .outputMode("append")
        .option("checkpointLocation", str(CKPT / "silver_no_watermark"))
        .trigger(availableNow=True)
        .toTable("silver_no_watermark")
    )
    q.awaitTermination()
    print("stream completed")
except Exception as e:
    print(type(e).__name__)
    print(str(e)[:600])

**Q6b.** Compare the total `readings` in `silver_readings_10min` against the row count
in `bronze_readings`. How many records did the 15-minute watermark drop? Your generator
produced records up to 45 minutes late — is the number consistent with that?

<details>
<summary><b>Answer</b></summary>

The "dropped" figure is large, anywhere from about a quarter to over 90% depending on how quickly
you ran the notebook, and it is tempting to blame the 45-minute-late readings. **It isn't them.** Run
the cell after the counts: the watermark dropped **zero** late rows. Every missing row is in a
window that is **not finished yet**.

- The watermark only advances **between** micro-batches: it is computed from the maximum event time
  of the batches already processed. Section 6 read all of bronze in its first batch, when the
  watermark was still at its starting value, so every late reading was accepted.
- `append` mode emits a window only once the watermark passes its end. After the run, the watermark
  is *latest event − 15 min*, so the last two or three 10-minute windows are still open. Their rows
  are held **in state**, and they will be written when later data moves the watermark on. The live
  run in 7c shows this happening.

So "bronze minus silver" mixes two very different things:

| | what it is | permanent? |
|---|---|---|
| rows in open windows | normal append-mode latency | no, they are emitted later |
| `numRowsDroppedByWatermark` | readings whose window had already closed | **yes, lost** |

True drops happen when a late reading arrives in a *later* batch, after its window has closed. With
readings up to 45 minutes late and a 15-minute watermark, that is the readings about 15–45 minutes
late (depending on where they fall in their window). You will see them in 7c, in the
`rows_dropped_by_watermark` column: typically 15–20 of every 150 readings, about 10–13% of each
batch. That is in the right range for the generator's settings: 30% of readings are late, and
roughly half of those are late enough to miss their window.

</details>

In [ ]:
bronze_n = spark.table("bronze_readings").count()
silver_n = spark.table("silver_readings_10min").agg(F.sum("readings")).first()[0]  # type: ignore
print(f"bronze: {bronze_n:,}")
print(f"silver: {silver_n or 0:,}")
print(f"dropped: {bronze_n - (silver_n or 0):,}  ({100 * (bronze_n - (silver_n or 0)) / bronze_n:.1f}%)")

In [ ]:
# Where did the missing rows go? Split them into "still in state" and "really dropped".
progress = q_silver.lastProgress
watermark = progress["eventTime"].get("watermark", "1970-01-01T00:00:00.000Z")
dropped_late = sum(
    op.get("numRowsDroppedByWatermark", 0) for p in q_silver.recentProgress for op in p["stateOperators"]
)
in_open_windows = (
    spark.table("bronze_readings")
    .withColumn("window_end", F.window(F.to_timestamp("event_time"), "10 minutes")["end"])
    .filter(F.col("window_end") > F.lit(watermark).cast("timestamp"))
    .count()
)
print(f"watermark after the run:            {watermark}")
print(f"rows in windows not yet closed:     {in_open_windows:,}")
print(f"rows dropped as too late:           {dropped_late:,}")
print(f"sum (should equal 'dropped' above): {in_open_windows + dropped_late:,}")

**Q6c.** That drop rate is a data quality incident nobody would notice. Sketch the
monitoring you would add. (Lecture 11 covers the machinery; here just say what you
would measure and what threshold would page someone.)

<details>
<summary><b>Answer</b></summary>

**Measure** (all of this is available per batch from a `StreamingQueryListener`, see 7b):

1. **Late-drop ratio** = `numRowsDroppedByWatermark / numInputRows` per stateful query. This is
   the number that directly represents lost data.
2. **Event-time lag distribution**: `_ingested_at − event_ts`, p50/p95/p99 per device or source,
   computed in a small side query on bronze. It tells you whether the 15-minute watermark still fits
   reality, and it moves *before* drops start.
3. **Reconciliation**: for each closed window, compare `SUM(readings)` in silver with
   `COUNT(*)` in bronze. It catches loss from any cause, not only the watermark.
4. **Freshness**: the time since the last committed batch, and watermark vs. wall clock. A dead or
   stuck stream drops nothing and looks healthy on every other metric.

**Thresholds** (a starting point, to be tuned against real traffic):
- **Page** when the late-drop ratio is above **1% over a 30-minute rolling window**, or when there
  has been no committed batch for 3× the trigger interval.
- **Ticket** (not page) on any drops for a day, or when p99 lag is above 80% of the watermark delay.
  That means the watermark needs widening before it starts losing data.
- Don't page on a single batch; see Q7b.

</details>

## 7. Monitoring a live stream

Everything so far used `availableNow`. Now run streams continuously and watch them three ways:

- **7a. Polling (pull)** — your code asks a query for `lastProgress` whenever it wants to.
- **7b. A listener (push)** — Spark calls your `StreamingQueryListener` after every micro-batch of
  *every* query in the session. This is how metrics get shipped to a monitoring system.
- **7c. A dashboard** — the listener also writes each batch's metrics to a CSV, and a small
  Streamlit app plots them live, in a separate process.

**Locally** the built-in live view is the **Structured Streaming** tab of the Spark UI at
<http://localhost:4040>. Keep it open while the cells run.

> **☁️ On Databricks** a live dashboard (input vs processing rate, batch duration) appears under
> every cell that starts a stream. **Serverless compute**, including Free Edition, accepts only
> `availableNow` triggers, so the `processingTime="10 seconds"` queries below are rejected
> there. Either use a classic cluster, or run one `availableNow` query per round of the loop.
> The Databricks version of this lab does the latter automatically.

### 7a. Polling `lastProgress`

In [ ]:
from pyspark_demo.streaming import stop_query

stop_query(spark, "bronze_live")  # left running if an earlier run of this cell was interrupted

q_live = (
    bronze_stream.writeStream.format("delta")
    .queryName("bronze_live")
    .outputMode("append")
    .option("checkpointLocation", str(CKPT / "bronze"))
    .option("mergeSchema", "true")
    .trigger(processingTime="10 seconds")
    .toTable("bronze_readings")
)

try:
    for i in range(6):
        drop_batch(150)
        time.sleep(10)
        p = q_live.lastProgress
        if p:
            print(
                f"batch {p['batchId']}: {p['numInputRows']:>5} rows  "
                f"in {p.get('inputRowsPerSecond', 0):>7.1f}/s  "
                f"out {p.get('processedRowsPerSecond', 0):>7.1f}/s  "
                f"{p['durationMs'].get('triggerExecution', 0)} ms"
            )
finally:
    q_live.stop()  # also when you interrupt the loop; 7c starts it again on the same checkpoint

Polling has two weaknesses. You only see the batch that happened to be the last one when you
looked, so batches in between are lost. And you need one loop per query.

### 7b. A custom `StreamingQueryListener`

`SimpleMetricsListener` lives in [`pyspark_demo/streaming.py`](../../pyspark_demo/streaming.py). It
prints a line for every start, progress and stop event. Given `csv_path`, it also appends one
row per micro-batch to a CSV file. Besides the rates, each row records the watermark, the state
size and `rows_dropped_by_watermark`, which is the number behind Q6b.

The callbacks run on the driver after each batch, so they must be quick and must never raise.

> **☁️ On Databricks** the listener is the same class, but it cannot write to a CSV in a Volume, since
> Unity Catalog volumes don't support appending to a file. Instead the Databricks version keeps the rows
> in memory and, **after** the run, saves them to a `stream_metrics` Delta table. It does not write
> the table from inside the callback, because that would block the listener thread. On serverless
> compute, a Python listener may not receive events at all; `query.recentProgress` has the same
> numbers.

Register the listener once. It then sees every query started in this session, including the two
below.

In [ ]:
from pyspark_demo.streaming import SimpleMetricsListener

METRICS_CSV = STREAMING_BASE / "metrics" / "progress.csv"  # wiped with STREAMING_BASE in setup

# Re-running this cell must not register a second listener, or every line prints twice.
if "metrics_listener" in globals():
    spark.streams.removeListener(metrics_listener)  # type: ignore  # noqa: F821

metrics_listener = SimpleMetricsListener(csv_path=METRICS_CSV)
spark.streams.addListener(metrics_listener)
print(f"listener registered, writing {METRICS_CSV}")

### 7c. A live dashboard

Run two queries at once: `bronze_live` ingests the landing folder, and `silver_live` is the
watermarked aggregation from section 6 running continuously on top of it. 30% of the new
readings are up to 45 minutes late, so the watermark will drop some of them.

**Before running the next cell**, start the dashboard in a terminal from the repo root:

```bash
uv run streamlit run labs/L5/monitor_dashboard.py
```

Open <http://localhost:8501>. The notebook cell blocks for about two minutes while it runs, which
is why the dashboard is a separate process that only reads `data/l5/metrics/progress.csv`.
Watch the charts update as batches complete.

> **☁️ On Databricks** there is no terminal next to the notebook, so there is no Streamlit here.
> The Databricks version charts the `stream_metrics` table with the notebook's built-in
> visualisations, and alerts can be SQL queries on that table (see Q7b). If you want this exact
> dashboard there, `monitor_dashboard.py` could be deployed as a **Databricks App** that reads the
> table instead of the CSV.

In [ ]:
stop_query(spark, "bronze_live", "silver_live")

q_bronze_live = (
    bronze_stream.writeStream.format("delta")
    .queryName("bronze_live")
    .outputMode("append")
    .option("checkpointLocation", str(CKPT / "bronze"))
    .option("mergeSchema", "true")
    .trigger(processingTime="10 seconds")
    .toTable("bronze_readings")
)

q_silver_live = (
    silver_stream.writeStream.format("delta")  # the watermarked aggregation from section 6
    .queryName("silver_live")
    .outputMode("append")
    .option("checkpointLocation", str(CKPT / "silver"))
    .trigger(processingTime="10 seconds")
    .toTable("silver_readings_10min")
)

# No polling here: the listener prints and logs every batch of both queries on its own.
try:
    for i in range(12):
        drop_batch(150, late_fraction=0.3, late_minutes=45)
        time.sleep(10)
finally:
    q_bronze_live.stop()
    q_silver_live.stop()

**Q7a.** If `inputRowsPerSecond` were consistently higher than `processedRowsPerSecond`,
what would happen over the next hour? Name two things you could change.

<details>
<summary><b>Answer</b></summary>

The stream **falls behind and never catches up**. Each trigger finds more new files than the last
batch could process, so batches get bigger and slower. Once a batch takes longer than the
10-second interval, the next one starts immediately, and the query is permanently busy. The
**end-to-end latency** from event to table grows steadily over the hour, the backlog in `LANDING`
grows with it, and one oversized batch can eventually hit out-of-memory or a timeout. Nothing
errors along the way: the dashboard just gets later and later.

Things to change:
- **Bound the batch size** with `maxFilesPerTrigger` / `maxBytesPerTrigger` (on Auto Loader,
  `cloudFiles.maxFilesPerTrigger`). Batches stay a steady size, and the backlog is worked through
  instead of arriving as one huge batch.
- **Add parallelism**: more cores or executors, and `spark.sql.shuffle.partitions` sized to them
  (we set 4 for a laptop).
- **Make each batch cheaper**: fewer or larger input files, less state (for example, a tighter
  watermark), the RocksDB state store for large state, and no expensive per-row UDFs.
- **Trigger less often**. Each batch has a fixed cost, most of the ~1 s you saw here (listing,
  Delta commit). A longer interval spreads that cost over more rows, trading latency for throughput.

</details>

**Q7b.** Go back to Q6c. Which column of `progress.csv` would you alert on, over what time
window, and at what threshold? Would you page on one bad batch?

<details>
<summary><b>Answer</b></summary>

Alert on **`rows_dropped_by_watermark`** of the stateful query (`silver_live`), as a **ratio**:
`sum(rows_dropped_by_watermark) / sum(num_input_rows)` over a rolling **30-minute** window, with a
page above **1%**, as in Q6c. A raw count means nothing without the volume it came from.

**Not on one bad batch.** A single spike is usually a device coming back online and flushing its
offline buffer: real data, but expected and self-limiting. Paging on it trains people to ignore
the pager. Require the ratio to hold over the window, or over several consecutive batches.

Two cheaper alerts belong alongside it:
- **no new rows in `progress.csv` for 3× the trigger interval**: the stream is dead or stuck. The
  listener's `onQueryTerminated` with an exception is the push version of this.
- **`input_rows_per_sec` above `processed_rows_per_sec` for 10+ minutes**: the backlog from Q7a
  (the dashboard's warning badge).

</details>

## Stop everything

**Run this cell.** A forgotten stream keeps a background thread polling the landing folder
(on Databricks it silently consumes your quota).

In [ ]:
for q in spark.streams.active:
    print(f"stopping {q.name or q.id}")
    q.stop()
print(f"{len(spark.streams.active)} active streams remaining")

if "metrics_listener" in globals():
    spark.streams.removeListener(metrics_listener)
    del metrics_listener
    print("listener removed")

In [ ]:
# Optional cleanup
# Databricks: dbutils.fs.rm(base, recurse=True)
# for t in ["bronze_readings", "silver_readings_10min"]:
#     spark.sql(f"DROP TABLE IF EXISTS {t}")
# shutil.rmtree(STREAMING_BASE, ignore_errors=True)
# spark.stop()